# 60 — Check a USDM study definition against the published model

Point `STUDY_JSON` below at any USDM v4 study definition (the JSON that
DDF-compliant systems exchange), set `BASE_IRI` to the namespace the
study's objects are named in (decision D7 in `../docs/iri-and-governance.md`),
and run all cells. Three checks:

1. **Context conformance** — every `instanceType` is a model class, every
   serialization key resolves to a model attribute, every id-based
   cross-reference resolves to an object in the document.
2. **Structural SHACL** (`usdm_v4.shapes.ttl`) — cardinality, datatypes,
   ranges, closedness.
3. **Terminology SHACL** (`usdm_v4.shapes-ct.ttl`) — the 25 DDF-native
   value sets, severity from CDISC's published extensibility flags. The
   other 20 codelist bindings in USDM are present as deactivated shapes;
   the run reports which of them the document actually uses.

What this is **not**: a regulatory or protocol-quality judgment.
Terminology coverage is the DDF-native value sets only — codelists backed
by external terminology packages (SDTM/Protocol Terminology, MedDRA,
ISO, ...) pass no value check here. `conforms=True` on the terminology
layer means "no finding among the 25 checked codelists"; the table of
unchecked values says what it does not mean. See `../docs/shacl-design.md`.

This notebook reports findings; it asserts nothing about them. It is an
optional checker, not part of the generation pipeline — the pipeline's
regression baselines live in `30_validate.ipynb`.

Requires the deliverables at the repo root (run notebooks 20, 40, 50
first, or use a release checkout) and the default input requires
`30_validate.ipynb` to have fetched the pilot study once.


## 1. Input


In [1]:
import json
from pathlib import Path

STUDY_JSON = "<repo>/usdm-master-protocol/output_cpmp/step3/cpmp_usdm.json"
BASE_IRI = "https://kerfors.github.io/early-clinical-design-research/usdm/cpmp/"

if not BASE_IRI.startswith(("http://", "https://")) or not BASE_IRI.endswith("/"):
    raise ValueError(
        "BASE_IRI must be an absolute http(s) instance namespace ending in '/' -"
        " the context declares no @base (docs/iri-and-governance.md, D7)"
    )

if not Path(STUDY_JSON).exists():
    raise FileNotFoundError(
        f"{STUDY_JSON} not found — for the default pilot study, run"
        " notebooks/30_validate.ipynb once, or set STUDY_JSON to your own file."
    )

with open(STUDY_JSON) as f:
    document = json.load(f)

if "study" not in document:
    raise ValueError("expected the USDM API wrapper with a 'study' key")
usdm_version = document.get("usdmVersion")
if usdm_version != "4.0.0":
    raise ValueError(
        f"usdmVersion is {usdm_version!r} — this checker covers USDM 4.0.0 only"
    )
study = document["study"]

print(f"input:       {STUDY_JSON}")
print(f"usdmVersion: {usdm_version}")
print(f"system:      {document.get('systemName')} {document.get('systemVersion')}")

id_locations = {}


def walk_ids(node, path):
    if isinstance(node, dict):
        if isinstance(node.get("id"), str):
            id_locations.setdefault(node["id"], []).append(path)
        for k, v in node.items():
            walk_ids(v, f"{path}.{k}")
    elif isinstance(node, list):
        for i, v in enumerate(node):
            walk_ids(v, f"{path}[{i}]")


walk_ids(study, "study")
duplicates = {k: v for k, v in id_locations.items() if len(v) > 1}
if duplicates:
    detail = "; ".join(f"{k}: {', '.join(v)}" for k, v in duplicates.items())
    raise ValueError(
        f"{len(duplicates)} duplicate object id(s) - RDF would merge them into"
        f" one node before any shape runs: {detail}"
    )
print(f"{len(id_locations)} distinct object ids, no duplicates")


input:       <repo>/usdm-master-protocol/output_cpmp/step3/cpmp_usdm.json
usdmVersion: 4.0.0
system:      early-clinical-design-research/12_cpmp_usdm 2026-10-25
1716 distinct object ids, no duplicates


## 2. Context conformance


In [2]:
import rdflib
from rdflib.namespace import OWL, RDF

CONTEXT_PATH = "../usdm_v4.context.jsonld"
TTL_PATH = "../usdm_v4.ttl"
for p in (CONTEXT_PATH, TTL_PATH):
    if not Path(p).exists():
        raise FileNotFoundError(f"{p} missing — generate the deliverables first.")

with open(CONTEXT_PATH) as f:
    context = json.load(f)["@context"]

unknown_types = set()
unmapped = {}
null_ids = []
typed_objects = 0


def walk(node):
    global typed_objects
    if isinstance(node, dict):
        instance_type = node.get("instanceType")
        if isinstance(instance_type, str):
            typed_objects += 1
            if instance_type not in context:
                unknown_types.add(instance_type)
            else:
                scoped = context[instance_type]["@context"]
                for k in node:
                    if k not in scoped and k not in ("id", "instanceType"):
                        unmapped.setdefault(instance_type, set()).add(k)
            if node.get("id") is None:
                null_ids.append(instance_type)
        for k, v in node.items():
            if k != "@context":
                walk(v)
    elif isinstance(node, list):
        for v in node:
            walk(v)


walk(study)

study["@context"] = context
instances = rdflib.Graph()
instances.parse(
    data=json.dumps(study), format="json-ld",
    publicID=BASE_IRI,
)

off_base = sorted(
    str(s) for s in set(instances.subjects())
    if isinstance(s, rdflib.URIRef) and not str(s).startswith(BASE_IRI)
)
if off_base:
    raise ValueError(
        f"{len(off_base)} subject IRI(s) outside BASE_IRI - D7 requires every"
        f" instance node to be minted under the supplied base: {off_base[:5]}"
    )

ontology = rdflib.Graph()
ontology.parse(TTL_PATH, format="turtle")
from rdflib.namespace import DCTERMS

source = next(ontology.objects(
    rdflib.URIRef("https://w3id.org/cdisc/usdm/v4/"), DCTERMS.source), None)
if source is None or "/DDF-RA/" not in str(source):
    raise ValueError(
        "ontology header lacks a DDF-RA dcterms:source — cannot state what this check is pinned against"
    )
DDF_RA_TAG = str(source).split("/DDF-RA/")[1].split("/")[0]
print(f"checking against:     DDF-RA {DDF_RA_TAG} (from ontology dcterms:source)")
declared_props = (
    set(ontology.subjects(RDF.type, OWL.DatatypeProperty))
    | set(ontology.subjects(RDF.type, OWL.ObjectProperty))
)

ref_props = set()
for class_def in context.values():
    if isinstance(class_def, dict) and "@context" in class_def:
        for term in class_def["@context"].values():
            if term.get("@type") == "@id":
                ref_props.add(rdflib.URIRef(
                    term["@id"].replace("usdm:", "https://w3id.org/cdisc/usdm/v4/")
                ))

ref_links = 0
dangling = []
for p in ref_props:
    for s, o in instances.subject_objects(p):
        ref_links += 1
        if (o, RDF.type, None) not in instances:
            dangling.append((str(p).rsplit("/", 1)[1], str(o).rsplit("/", 1)[-1]))

print(f"typed objects:        {typed_objects}")
print(f"instance triples:     {len(instances)}")
print(f"unknown instanceTypes: {sorted(unknown_types) or 'none'}")
print(f"unmapped keys:        {({k: sorted(v) for k, v in unmapped.items()}) or 'none'}")
print(f"objects with null id: {null_ids or 'none'}")
print(f"cross-references:     {ref_links}, dangling: {len(dangling)}")
for prop, target in dangling[:20]:
    print(f"  dangling: {prop} -> {target}")


checking against:     DDF-RA v4.0.0 (from ontology dcterms:source)
typed objects:        1716
instance triples:     12507
unknown instanceTypes: none
unmapped keys:        none
objects with null id: none
cross-references:     3100, dangling: 0


## 3. SHACL — structural and terminology layers


In [3]:
import re
import pandas as pd
from pyshacl import validate
from rdflib.namespace import RDFS, SH

USDM = rdflib.Namespace("https://w3id.org/cdisc/usdm/v4/")

SHAPES = {
    "structural": "../usdm_v4.shapes.ttl",
    "terminology": "../usdm_v4.shapes-ct.ttl",
}
for p in SHAPES.values():
    if not Path(p).exists():
        raise FileNotFoundError(f"{p} missing — run notebooks/50_generate_shapes.ipynb first.")

rows = []
deactivated = []
for layer, path in SHAPES.items():
    shapes_graph = rdflib.Graph()
    shapes_graph.parse(path, format="turtle")
    conforms, report, _ = validate(
        data_graph=instances, shacl_graph=shapes_graph,
        inference="none", abort_on_first=False,
    )
    n = 0
    for result in report.subjects(SH.resultSeverity, None):
        n += 1
        path_value = report.value(result, SH.resultPath)
        focus = report.value(result, SH.focusNode)
        value = report.value(result, SH.value)
        value_str = str(value) if value is not None else ""
        rows.append({
            "layer": layer,
            "severity": str(report.value(result, SH.resultSeverity)).rsplit("#", 1)[1],
            "constraint": str(report.value(result, SH.sourceConstraintComponent)).rsplit("#", 1)[1].removesuffix("ConstraintComponent"),
            "path": str(path_value).rsplit("/", 1)[1] if path_value else "",
            "focus": str(focus).rsplit("/", 1)[-1],
            "value": value_str,
            "value_ncit": (f"http://purl.obolibrary.org/obo/NCIT_{value_str}"
                           if re.fullmatch(r"C\d+", value_str) else ""),
        })
    scope = ""
    if layer == "terminology":
        deactivated = sorted(
            (shapes_graph.value(s, SH.targetObjectsOf), shapes_graph.value(s, RDFS.seeAlso))
            for s in shapes_graph.subjects(SH.deactivated, rdflib.Literal(True))
        )
        n_shapes = sum(1 for _ in shapes_graph.subjects(RDF.type, SH.NodeShape))
        scope = (f"  scope: {n_shapes - len(deactivated)} bound codelists checked,"
                 f" {len(deactivated)} not checked (sh:deactivated)")
    print(f"{layer:<12} conforms={conforms}  findings={n}{scope}")

findings = pd.DataFrame(rows, columns=["layer", "severity", "constraint", "path", "focus", "value", "value_ncit"])
if len(findings):
    display(findings.sort_values(["layer", "severity", "path"]).reset_index(drop=True))
else:
    print("no findings — the study conforms to both layers")

unchecked_rows = []
for prop, see_also in deactivated:
    codes = []
    for obj in instances.objects(None, prop):
        code = instances.value(obj, USDM["Code-code"])
        if code is None:   # AliasCode: the code sits on standardCode
            code = instances.value(
                instances.value(obj, USDM["AliasCode-standardCode"]), USDM["Code-code"])
        codes.append(str(code))
    if codes:
        unchecked_rows.append({
            "property": str(prop).rsplit("/", 1)[1],
            "codelist": str(see_also).rsplit("/", 1)[1],
            "occurrences": len(codes),
            "distinct_codes": len(set(codes)),
            "codes": " ".join(sorted(set(codes))),
        })
unchecked = pd.DataFrame(unchecked_rows, columns=["property", "codelist", "occurrences", "distinct_codes", "codes"])
print(f"\n{int(unchecked['occurrences'].sum())} coded value(s) ({int(unchecked['distinct_codes'].sum())} distinct)"
      f" in this document are governed by a codelist this checker does not check"
      f" ({len(unchecked)} of {len(deactivated)} deactivated bindings used):")
if len(unchecked):
    display(unchecked.sort_values("property").reset_index(drop=True))


structural   conforms=False  findings=55
terminology  conforms=True  findings=0  scope: 25 bound codelists checked, 20 not checked (sh:deactivated)


,layer,severity,constraint,path,focus,value,value_ncit
0,structural,Violation,MinCount,GovernanceDate-geographicScopes,GovernanceDate_6,,
1,structural,Violation,MinCount,GovernanceDate-geographicScopes,GovernanceDate_5,,
2,structural,Violation,MinCount,GovernanceDate-geographicScopes,GovernanceDate_3,,
3,structural,Violation,MinCount,GovernanceDate-geographicScopes,GovernanceDate_1,,
4,structural,Violation,MinCount,GovernanceDate-geographicScopes,GovernanceDate_12,,
5,structural,Violation,MinCount,GovernanceDate-geographicScopes,GovernanceDate_4,,
6,structural,Violation,MinCount,GovernanceDate-geographicScopes,GovernanceDate_2,,
7,structural,Violation,MinCount,GovernanceDate-geographicScopes,GovernanceDate_7,,
8,structural,Violation,MinCount,GovernanceDate-geographicScopes,GovernanceDate_8,,
9,structural,Violation,MinCount,GovernanceDate-geographicScopes,GovernanceDate_9,,



0 coded value(s) (0 distinct) in this document are governed by a codelist this checker does not check (0 of 20 deactivated bindings used):


## 4. Reading the results

- **Violation** (structural): the study breaks a model rule — a missing
  required attribute, a wrong datatype, an undeclared key, a value of the
  wrong class. `MinCount` on a `1..*` attribute with an empty list is the
  classic case: the information may exist in prose while the required
  structured form is empty.
- **Violation** (terminology): a value outside a **non-extensible**
  codelist. Three readings, in rough order of likelihood: the value is
  wrong; a placeholder was never replaced; or the value comes from a
  newer or parallel CT release than the pinned DDF-RA tag — the
  toolchain running ahead of the published standard (e.g. C207646
  "Study Acronym", CDISC-defined, where DDF-RA v4.0.0 publishes the
  ICH-rooted C94108 for the same term). Check the finding's code in
  NCIt before treating it as a data error.
- **Warning** (terminology): a value outside an **extensible** codelist.
  Sponsor extensions are legitimate use; the warning flags them for
  review, nothing more.
- The `focus` column carries the object's `id` from your JSON, so
  findings can be traced back to the source document.
- **Unchecked values** (terminology): coded values on properties whose
  codelist is bound in USDM but whose members are not in the USDM
  deliverables (SDTM/Protocol Terminology). They are not findings and
  not passes — they were not looked at. Check them against the CT
  package your document was built with.

Worked examples (DDF-RA v4.0.0 published studies): the CDISC Pilot is
structurally clean but carries 14 terminology findings — placeholder
codes (`C12345`, `C99905x1`, ...) that were never replaced. The Eli Lilly
diabetes example is the mirror image: every code valid, one structural
violation — an amendment whose two changes are described in the
`summary` prose while the required `changes` list (`1..*`) sits empty.


## Provenance

Part of [kerfors/usdm-rdf](https://github.com/kerfors/usdm-rdf). Checks
against the deliverables generated by notebooks 20, 40, and 50 from the
DDF-RA tag pinned in `notebooks/10_fetch_yaml.ipynb`.


In [4]:
import pandas as pd
_out = Path(r'<repo>/usdm-master-protocol/output_cpmp/step3/validation')
pd.DataFrame(rows).to_csv(_out / 'shacl_findings.csv', index=False)
pd.DataFrame([{'class': k, 'key': v} for k, vs in unmapped.items() for v in sorted(vs)], columns=['class', 'key']).to_csv(_out / 'context_unmapped_keys.csv', index=False)
print('unknown types:', sorted(unknown_types))
print('findings:', len(rows))


unknown types: []
findings: 55
